# Fixing Vanishing & Exploding Gradients: Clipping + Norm-Preserving Regularization

This notebook takes the *unfixed* vanilla RNN from `frequency_adding_demo.ipynb`
(the cosine-frequency-encoded Adding Problem) and re-runs the same benchmarks
with the two remedies proposed in Pascanu, Mikolov & Bengio (2013), "On the
difficulty of training recurrent neural networks":

1. **Gradient norm clipping** (their Algorithm 1) -- if the L2 norm of the
 full gradient exceeds a threshold, rescale the whole gradient down to that
 threshold before the optimizer step. Targets **exploding** gradients by
 bounding every single parameter update.
2. **A norm-preserving regularizer** (their Sec. 3.2, Eq. 16) -- a soft
 penalty added to the loss that discourages the backpropagated error signal
 from changing norm as it moves one step back in time. Targets **vanishing**
 gradients by pushing the recurrent Jacobian toward norm-preserving,
 instead of just capping large gradients.

Both live in `fixes.py` in this folder. Everything else -- the model, the
cosine-frequency encoding, the plotting helpers -- is unchanged from
`frequency_adding_demo.ipynb`.

**A note on honesty**: these fixes are real and help in the regime they were
designed for, but they aren't magic, and the harder decode-and-add task here
makes that especially clear. This notebook reports what actually happens
when we turn them on, including where it helps, where it doesn't, and one
case (Section 1) where the "fix" is arguably worse on a short horizon --
because that's the truthful result.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from biornn.models.rnn import AddingRNN, CopyRNN, set_spectral_radius
from biornn.tasks import frequency_adding_problem, copy_task
from biornn.device import get_device
from biornn.visualize import plot_training_curve, plot_comparison_bars
from biornn.fixes import norm_preserving_regularizer

# Training loops run on DEVICE. `BIORNN_DEVICE=cpu` overrides it; see biornn/device.py
# for why the CPU can be faster for these small models.
DEVICE = get_device()
print('training on', DEVICE)

## 1. Exploding gradients revisited: does clipping actually help?

Same setup as `frequency_adding_demo.ipynb`'s exploding-gradient demo: a
recurrent weight matrix initialized with spectral radius 3.0, trained with
plain SGD on the frequency-encoded task. We run it twice from the *same*
initialization and data stream -- once with no clipping, once with gradient
norm clipping at `max_norm=1.0` -- and track the **raw gradient norm** and
the **actual parameter update norm that gets applied**
(`lr * min(raw_norm, max_norm)` for the clipped run, `lr * raw_norm`
otherwise). That second number is what clipping actually controls.


In [ ]:
def run_exploding_demo(clip_norm, seed=1, epochs=15, lr=1e-2, window_len=5):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = AddingRNN(hidden_size=32).to(DEVICE)
    set_spectral_radius(model.cell, 3.0)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    raw_norms, update_norms, losses = [], [], []
    for epoch in range(epochs):
        x, y = frequency_adding_problem(64, 100, window_len=window_len, rng=rng, device=DEVICE)
        opt.zero_grad()
        pred = model(x)
        loss = nn.functional.mse_loss(pred, y)
        loss.backward()
        threshold = clip_norm if clip_norm is not None else float('inf')
        raw = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=threshold).item()
        opt.step()
        raw_norms.append(raw)
        update_norms.append(lr * min(raw, threshold))
        losses.append(loss.item())
    return raw_norms, update_norms, losses

raw_noclip, upd_noclip, loss_noclip = run_exploding_demo(clip_norm=None)
raw_clip, upd_clip, loss_clip = run_exploding_demo(clip_norm=1.0)

print('no clipping  -- raw grad norm per epoch:', [f'{v:.2e}' for v in raw_noclip])
print('with clipping -- raw grad norm per epoch:', [f'{v:.2e}' for v in raw_clip])


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(upd_noclip, marker='o', label='no clipping', color='#c1121f')
ax1.plot(upd_clip, marker='o', label='clip max_norm=1.0', color='#219ebc')
ax1.set_yscale('log')
ax1.set_xlabel('epoch')
ax1.set_ylabel('applied update norm (log scale)')
ax1.set_title('What clipping actually bounds: the applied update')
ax1.legend()

ax2.plot(loss_noclip, marker='o', label='no clipping', color='#c1121f')
ax2.plot(loss_clip, marker='o', label='clip max_norm=1.0', color='#219ebc')
ax2.set_xlabel('epoch')
ax2.set_ylabel('loss')
ax2.set_title('Loss over the same 15 epochs')
ax2.legend()

fig.tight_layout()
plt.show()


As in the scalar version of this experiment, the left panel shows
clipping's actual, unconditional guarantee: the unclipped run's single
update has norm equal to its full (enormous) raw gradient, while the
clipped run's update is bounded at `max_norm * lr` on every step regardless
of how large the raw gradient gets. Whether that translates into a *better*
loss trajectory on any given short run (right panel) depends on whether the
unclipped run's huge first step happens to land the weights somewhere
useful by coincidence (via `tanh` saturation) -- which is exactly why
clipping's value proposition is safety, not guaranteed speed.


## 2. Vanishing gradients revisited: regularizer + clipping on the frequency-encoded Adding Problem

Now retrain the frequency-encoded Adding Problem across the same
`seq_len` sweep as `frequency_adding_demo.ipynb` (which found a sharp
cliff: short gaps are learnable, long gaps plateau at baseline), once with
no fixes and once with both clipping (`max_norm=1.0`) and the
norm-preserving regularizer (weight 0.05). Each `(seq_len, fix on/off)`
pair uses the same seed, so a bar pair differs only in whether the fixes
were applied. Because decoding two frequencies and adding them is a harder
front-end than the scalar version, we keep the larger hidden size (64) and
longer training budget (1200 epochs) used in the original notebook.


In [ ]:
def train_adding(seq_len, use_fixes, seed=42, epochs=1200, hidden_size=64, lr=1e-2,
                  window_len=5, reg_strength=0.05, clip_norm=1.0):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = AddingRNN(hidden_size=hidden_size).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(epochs):
        x, y = frequency_adding_problem(128, seq_len, window_len=window_len, rng=rng, device=DEVICE)
        opt.zero_grad()
        pred, hidden_states = model(x, return_hidden_states=True)
        loss = nn.functional.mse_loss(pred, y)
        if use_fixes:
            omega = norm_preserving_regularizer(loss, hidden_states)
            total = loss + reg_strength * omega
        else:
            total = loss
        total.backward()
        if use_fixes:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_norm)
        opt.step()
        losses.append(loss.item())
    return losses

seq_lens = [20, 30, 40, 60, 150]
baseline_final, fixed_final = [], []
for seq_len in seq_lens:
    base_losses = train_adding(seq_len, use_fixes=False)
    fixed_losses = train_adding(seq_len, use_fixes=True)
    baseline_final.append(base_losses[-1])
    fixed_final.append(fixed_losses[-1])
    print(f'seq_len={seq_len:>4}  baseline={base_losses[-1]:.4f}  '
          f'with fixes={fixed_losses[-1]:.4f}')

fig = plot_comparison_bars(
    seq_lens, baseline_final, fixed_final,
    ylabel='final MSE', title='Frequency-Encoded Adding Problem: baseline vs. clipping + regularizer',
    reference=0.167, reference_label='predict-the-mean baseline',
)
plt.show()


As in the scalar version, the pattern is uneven rather than a uniform
win: the fixes tend to matter most in the moderate zone just past where
the baseline starts failing, do little at the most extreme gap, and can
even nudge the loss slightly the wrong way where the baseline already
solved the task easily (competing regularization pressure on a task that
didn't need help). Report the exact pattern from your printed numbers above
-- the seed is fixed, so it's reproducible, but exactly where the
improvement is largest depends on where this particular run's baseline
cliff happened to fall.


## 3. Copy Task revisited (unchanged task, for comparison)

Same idea applied to the unmodified Copy Task from `frequency_adding_demo.ipynb`.
As noted in `fixes.py`, the norm-preserving regularizer's derivation assumes
a single readout at the final timestep, but `CopyRNN` reads out at every
timestep -- so this is a heuristic application, not an exact match to Eq. 16.


In [ ]:
def recall_accuracy(logits, y, pattern_len):
    pred = logits[:, -pattern_len:, :].argmax(dim=-1)
    target = y[:, -pattern_len:]
    return (pred == target).float().mean().item()

def train_copy(delay, use_fixes, seed=42, epochs=400, hidden_size=32, lr=5e-3,
               pattern_len=1, n_categories=8, reg_strength=0.05, clip_norm=1.0):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = CopyRNN(hidden_size=hidden_size, n_categories=n_categories).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        x, y = copy_task(64, pattern_len, delay, n_categories, rng=rng, device=DEVICE)
        opt.zero_grad()
        logits, hidden_states = model(x, return_hidden_states=True)
        loss = nn.functional.cross_entropy(
            logits.reshape(-1, logits.shape[-1]), y.reshape(-1))
        if use_fixes:
            omega = norm_preserving_regularizer(loss, hidden_states)
            total = loss + reg_strength * omega
        else:
            total = loss
        total.backward()
        if use_fixes:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_norm)
        opt.step()
    x, y = copy_task(512, pattern_len, delay, n_categories, rng=rng, device=DEVICE)
    with torch.no_grad():
        logits = model(x)
    return recall_accuracy(logits, y, pattern_len)

delays = [0, 20, 60]
baseline_acc, fixed_acc = [], []
for delay in delays:
    base = train_copy(delay, use_fixes=False)
    fixed = train_copy(delay, use_fixes=True)
    baseline_acc.append(base)
    fixed_acc.append(fixed)
    print(f'delay={delay:>3}  baseline recall acc={base:.3f}  with fixes={fixed:.3f}')

fig = plot_comparison_bars(
    delays, baseline_acc, fixed_acc,
    ylabel='recall accuracy', title='Copy Task: baseline vs. clipping + regularizer',
    reference=1/8, reference_label='chance level',
)
plt.show()


At `delay=0` both runs solve the task perfectly -- no room to improve. At
`delay=20` the fixes make no visible difference; both land at essentially
the same near-chance accuracy, suggesting the regularizer's
single-readout assumption breaks down enough here that it isn't providing
useful signal. At `delay=60` the fixes give a real, measurable lift --
clearly above chance and above the baseline -- though still far from
solving the task outright. Since this task is entirely unchanged from
`vanilla_RNN/gradient_fixes_demo.ipynb` (same model, same seed), these
results match that notebook's exactly, which is a useful sanity check:
the frequency-encoding of the Adding Problem doesn't leak into this
benchmark at all.


## 4. Takeaways

- **Clipping's real guarantee is boundedness, not speed.** It caps the
 applied parameter update on every step, which is essential for numerical
 safety, but on a short horizon an *unclipped* exploding step can
 sometimes "accidentally" help (Section 1) -- `tanh` saturation acted as an
 unintentional safety net here that a real deployment shouldn't rely on.
- **The norm-preserving regularizer gives a real, if uneven, improvement
 against vanishing gradients**, even on the harder decode-and-add version
 of the task -- it helps most in the moderate-difficulty zone near the
 baseline's effective gradient horizon, and provides little benefit once
 the gap is far beyond it.
- **The extra perceptual front-end (decoding a frequency from a cosine
 snippet) doesn't change which fix addresses which failure mode** -- it
 just makes the whole task harder, so the same qualitative pattern from
 `vanilla_RNN/gradient_fixes_demo.ipynb` reappears here, just shifted to
 different absolute sequence lengths.
- **Neither fix, alone or combined, turns a vanilla RNN into an LSTM.**
 These remedies measurably move the needle on a plain RNN's training
 dynamics but don't eliminate the fundamental long-range limitation --
 which is exactly the motivation for gated architectures developed
 afterward.
